# Supervised ViT-B/16 Linear Probe - Defect Data Sweep

Binary classifier built on a frozen ViT-B/16 backbone (same as the best zero-defect PatchCore model).
A linear head is trained on normal wafers + a sweep of labeled defect samples.

**Holdout classes:** `Scratch` and `Loc` are completely withheld from training and validation.
All 8 defect types appear in the test set so overall AUROC/F1 is directly comparable
to the zero-defect PatchCore ViT-B/16 result (F1=0.595, AUROC=0.956).

**Purpose:** demonstrate that a supervised classifier improves on *seen* defect types
as more labeled data is added, but consistently fails to detect *unseen* types (Scratch, Loc)
regardless of how much seen-class data is used - the core limitation of the supervised paradigm.


## Imports and Paths


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
cwd = Path.cwd().resolve()
REPO_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'src' / 'wafer_defect').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not find repo root containing src/wafer_defect')
EXPERIMENT_DIR = REPO_ROOT / 'experiments' / 'anomaly_detection_defect' / 'supervised_sweep' / 'vit_b16' / 'x224' / 'main'
TRAIN_CONFIG = EXPERIMENT_DIR / 'train_config.toml'
ARTIFACT_DIR = EXPERIMENT_DIR / 'artifacts' / 'supervised_vit_sweep'
RESULTS_DIR = ARTIFACT_DIR / 'results'
PLOTS_DIR = ARTIFACT_DIR / 'plots'
RUNNER_SCRIPT = REPO_ROOT / 'scripts' / 'run_supervised_sweep_vit_b16_x224.py'
PATCHCORE_AUROC = 0.956
PATCHCORE_F1 = 0.595
PATCHCORE_AUPRC = 0.671
RERUN = False
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repo root    : {REPO_ROOT}')
print(f'Artifact dir : {ARTIFACT_DIR}')
print(f'Results dir  : {RESULTS_DIR} (exists={RESULTS_DIR.exists()})')


In [ ]:
RETRAIN = False
print(f'RETRAIN = {RETRAIN}')


## Optional Rerun

Normally the sweep is run on Modal and results are downloaded to `artifacts/`.


In [ ]:
if RERUN:
    cmd = [sys.executable, '-u', str(RUNNER_SCRIPT), '--config', str(TRAIN_CONFIG)]
    print('Launching sweep:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=REPO_ROOT)
else:
    print('RERUN=False - loading saved results.')
    if not (RESULTS_DIR / 'sweep_summary.json').exists():
        print('WARNING: sweep_summary.json not found.\nRun the Modal app first:  modal run --detach modal_apps/supervised_sweep_vit_b16_x224/app.py::main')


## Load Sweep Results


In [ ]:
summary_path = RESULTS_DIR / 'sweep_summary.json'
SWEEP_RESULTS_AVAILABLE = False
sweep_summary = {}
cfg = {}
holdout_types = []
results = []
df = pd.DataFrame()
if not summary_path.exists():
    print(f'[WARNING] sweep_summary.json was not found at {summary_path}.')
    print('[WARNING] The saved sweep review will be skipped until local artifacts are available.')
else:
    sweep_summary = json.loads(summary_path.read_text(encoding='utf-8'))
    cfg = sweep_summary['config']
    holdout_types = cfg['holdout_defect_types']
    results = sweep_summary['sweep_results']
    df = pd.DataFrame([{'fraction': r['fraction'], 'n_defects_train': r['n_defects_train'], 'auroc': r['auroc'], 'auprc': r['auprc'], 'f1': r['f1'], 'precision': r['precision'], 'recall': r['recall'], 'best_sweep_f1': r['best_sweep_f1']} for r in results])
    SWEEP_RESULTS_AVAILABLE = not df.empty
if SWEEP_RESULTS_AVAILABLE:
    print(f'Holdout classes (unseen during training): {holdout_types}')
    print(f"Train defect pool size: {cfg['train_defect_pool_size']}")
    print(f"Total labeled defects: {cfg['total_labeled_defects']}")
    print()
    display(df.set_index('n_defects_train').round(3))
else:
    print('[WARNING] No supervised sweep rows are available to display in this notebook run.')


## Sweep Performance vs Zero-Defect Baseline

AUROC, AUPRC and F1 evaluated on the same benchmark as zero-defect PatchCore ViT-B/16
(5,000 test normals + 250 test defects across all 8 types).


In [ ]:
if SWEEP_RESULTS_AVAILABLE:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metrics = [('auroc', 'AUROC', PATCHCORE_AUROC), ('auprc', 'AUPRC', PATCHCORE_AUPRC), ('f1', 'F1', PATCHCORE_F1)]
    for ax, (col, label, baseline) in zip(axes, metrics):
        ax.plot(df['n_defects_train'], df[col], marker='o', linewidth=2, color='steelblue', label='Supervised (seen classes)')
        ax.axhline(baseline, color='tomato', linestyle='--', linewidth=1.5, label=f'Zero-defect PatchCore ({baseline:.3f})')
        ax.set_xlabel('Defect training samples (seen classes)')
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.legend(fontsize=8)
        ax.set_ylim(0, 1.05)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    plt.suptitle(f'Supervised ViT-B/16 sweep vs zero-defect PatchCore ViT-B/16\n(holdout classes excluded from training: {holdout_types})', fontsize=11)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'sweep_performance.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Sweep performance plots are unavailable because the saved supervised sweep summary is missing.')
else:
    print('Sweep performance plots are unavailable because the saved supervised sweep summary is missing.')
else:
    print('Sweep performance plots are unavailable because the saved supervised sweep summary is missing.')


## Per-Class Recall Across the Sweep

Seen classes (used during training) vs holdout classes (never seen during training).
Holdout class recall should remain near zero regardless of sweep size.


In [ ]:
if SWEEP_RESULTS_AVAILABLE:
    per_class_rows = []
    for r in results:
        for dtype, stats in r['per_class_recall'].items():
            per_class_rows.append({'n_defects_train': r['n_defects_train'], 'defect_type': dtype, 'recall': stats['recall'], 'seen_in_training': stats['seen_in_training'], 'n': stats['n']})
    per_class_df = pd.DataFrame(per_class_rows)
    pivot = per_class_df.pivot(index='defect_type', columns='n_defects_train', values='recall').round(3)
    pivot.index.name = 'Defect type'
    seen_types = sorted(per_class_df[per_class_df['seen_in_training']]['defect_type'].unique())
    holdout_shown = sorted(per_class_df[~per_class_df['seen_in_training']]['defect_type'].unique())
    print('Per-class recall by number of training defects')
    print(f'  Seen classes (trained on): {seen_types}')
    print(f'  Holdout classes (unseen):  {holdout_shown}')
    print()
    display(pivot.style.background_gradient(cmap='RdYlGn', axis=None, vmin=0, vmax=1))
    pivot.to_csv(RESULTS_DIR / 'per_class_recall_sweep.csv')
else:
    print('Per-class recall tables are unavailable because the saved supervised sweep summary is missing.')
else:
    print('Per-class recall tables are unavailable because the saved supervised sweep summary is missing.')
else:
    print('Per-class recall tables are unavailable because the saved supervised sweep summary is missing.')


## Seen vs Holdout Recall - Side-by-Side

Each bar group is one sweep point. Seen classes improve with more data;
holdout classes stay near zero regardless.


In [ ]:
if SWEEP_RESULTS_AVAILABLE:
    n_defects_vals = sorted(per_class_df['n_defects_train'].unique())
    avg_seen = [per_class_df[(per_class_df['n_defects_train'] == n) & per_class_df['seen_in_training']]['recall'].mean() for n in n_defects_vals]
    avg_holdout = [per_class_df[(per_class_df['n_defects_train'] == n) & ~per_class_df['seen_in_training']]['recall'].mean() for n in n_defects_vals]
    x = np.arange(len(n_defects_vals))
    width = 0.35
    labels = [f'{n:,}' for n in n_defects_vals]
    fig, ax = plt.subplots(figsize=(10, 4))
    bars_seen = ax.bar(x - width / 2, avg_seen, width, label='Seen classes (avg)', color='steelblue', alpha=0.85)
    bars_holdout = ax.bar(x + width / 2, avg_holdout, width, label='Holdout classes (avg)', color='tomato', alpha=0.85)
    ax.set_xlabel('Defect training samples')
    ax.set_ylabel('Average recall')
    ax.set_title(f'Average recall: seen vs holdout defect classes\nHoldout ({holdout_types}): never seen during training')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.bar_label(bars_seen, fmt='%.2f', fontsize=8, padding=2)
    ax.bar_label(bars_holdout, fmt='%.2f', fontsize=8, padding=2)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'seen_vs_holdout_recall.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Seen-vs-holdout recall plots are unavailable because the saved supervised sweep summary is missing.')
else:
    print('Seen-vs-holdout recall plots are unavailable because the saved supervised sweep summary is missing.')
else:
    print('Seen-vs-holdout recall plots are unavailable because the saved supervised sweep summary is missing.')


## Per-Class Recall at Largest Sweep Point

Full breakdown at the highest defect training count, showing the contrast between
seen and holdout class recall most clearly.


In [ ]:
if SWEEP_RESULTS_AVAILABLE:
    max_n = max(n_defects_vals)
    final_df = per_class_df[per_class_df['n_defects_train'] == max_n].sort_values('recall', ascending=False)
    colors = ['steelblue' if row['seen_in_training'] else 'tomato' for _, row in final_df.iterrows()]
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(final_df['defect_type'], final_df['recall'], color=colors, alpha=0.85)
    ax.bar_label(bars, fmt='%.2f', fontsize=9, padding=2)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor='steelblue', alpha=0.85, label='Seen during training'), Patch(facecolor='tomato', alpha=0.85, label=f'Holdout - never seen ({holdout_types})')])
    ax.set_xlabel('Defect type')
    ax.set_ylabel('Recall')
    ax.set_title(f'Per-class recall at {max_n:,} training defects (supervised ViT-B/16)')
    ax.set_ylim(0, 1.15)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'per_class_recall_max_sweep.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Largest-sweep per-class recall plots are unavailable because the saved supervised sweep summary is missing.')
else:
    print('Largest-sweep per-class recall plots are unavailable because the saved supervised sweep summary is missing.')
else:
    print('Largest-sweep per-class recall plots are unavailable because the saved supervised sweep summary is missing.')


## Summary Table


In [ ]:
if SWEEP_RESULTS_AVAILABLE:
    summary_rows = []
    for r in results:
        seen_recall = np.mean([v['recall'] for v in r['per_class_recall'].values() if v['seen_in_training']])
        holdout_recall = np.mean([v['recall'] for v in r['per_class_recall'].values() if not v['seen_in_training']])
        summary_rows.append({'Defects (train)': f"{r['n_defects_train']:,}", 'Fraction': f"{r['fraction']:.3f}", 'AUROC': round(r['auroc'], 3), 'AUPRC': round(r['auprc'], 3), 'F1': round(r['f1'], 3), 'Best F1': round(r['best_sweep_f1'], 3), 'Avg seen recall': round(seen_recall, 3), 'Avg holdout recall': round(holdout_recall, 3)})
    summary_rows_display = [{'Defects (train)': '0 (zero-defect)', 'Fraction': '-', 'AUROC': PATCHCORE_AUROC, 'AUPRC': PATCHCORE_AUPRC, 'F1': PATCHCORE_F1, 'Best F1': '-', 'Avg seen recall': '-', 'Avg holdout recall': '- (generalises)'}] + summary_rows
    summary_table = pd.DataFrame(summary_rows_display)
    display(summary_table.set_index('Defects (train)'))
    summary_table.to_csv(RESULTS_DIR / 'sweep_summary_table.csv', index=False)
    print('Saved to sweep_summary_table.csv')
else:
    print('Summary tables are unavailable because the saved supervised sweep summary is missing.')
else:
    print('Summary tables are unavailable because the saved supervised sweep summary is missing.')
else:
    print('Summary tables are unavailable because the saved supervised sweep summary is missing.')


## Key Takeaways

**Improved performance with labeled data:**
As seen-class defect training samples increase, AUROC and F1 on the full benchmark improve.
With sufficient labeled data the supervised model can exceed the zero-defect PatchCore baseline
on known defect types.

**Failure on unseen defect types:**
The holdout classes (`Scratch`, `Loc`) are never shown during training. Their recall stays
near zero at every sweep point - adding more seen-class training data does not help.
The classifier has learned a decision boundary specific to the 6 seen defect types.
Any novel defect pattern that falls outside this boundary will be misclassified as normal.

**Advantage of the zero-defect paradigm:**
The zero-defect PatchCore model achieves non-trivial recall on all 8 defect types
without seeing any defect labels. By modelling only the normal distribution, it generalises
to arbitrary anomalies by definition - including defect types that have never been catalogued.


In [ ]:
if SWEEP_RESULTS_AVAILABLE:
    print('Saved plots:')
    for p in sorted(PLOTS_DIR.glob('*.png')):
        print(f'  {p}')
else:
    print('Saved-plot listing is unavailable because the saved supervised sweep summary is missing.')
else:
    print('Saved-plot listing is unavailable because the saved supervised sweep summary is missing.')
else:
    print('Saved-plot listing is unavailable because the saved supervised sweep summary is missing.')


## UMAP Review

This notebook now keeps the saved CLS-embedding UMAP review together with the main supervised sweep analysis. If the UMAP exports are not present locally, the section logs a warning and moves on.


In [ ]:
UMAP_OUTPUT_DIR = ARTIFACT_DIR / 'umap'
LEGACY_UMAP_OUTPUT_DIR = EXPERIMENT_DIR / 'umap' / 'artifacts'
umap_coords_candidates = [UMAP_OUTPUT_DIR / 'umap_coords.csv', LEGACY_UMAP_OUTPUT_DIR / 'umap_coords.csv']
UMAP_COORDS_PATH = next((path for path in umap_coords_candidates if path.exists()), None)
supervised_umap_df = None
if UMAP_COORDS_PATH is None:
    print('[WARNING] No saved supervised-sweep UMAP coordinate CSV was found.')
    for candidate in umap_coords_candidates:
        print('  -', candidate)
else:
    supervised_umap_df = pd.read_csv(UMAP_COORDS_PATH)
    print(f'Loaded UMAP coordinates from {UMAP_COORDS_PATH}')
    display(supervised_umap_df.head())
    display(supervised_umap_df['group'].value_counts(dropna=False).rename('count').to_frame())


In [ ]:
if supervised_umap_df is None or supervised_umap_df.empty:
    print('[WARNING] No UMAP-by-role plot was generated because the saved UMAP coordinates are unavailable.')
else:
    role_styles = {'train_normal': dict(color='#d1d5db', s=4, alpha=0.25, zorder=1, label='Train normal'), 'test_normal': dict(color='#4d908e', s=8, alpha=0.4, zorder=2, label='Test normal'), 'seen_defect': dict(color='#277da1', s=14, alpha=0.65, zorder=3, label='Seen defect'), 'holdout_scratch': dict(color='#e63946', s=22, alpha=0.85, zorder=5, label='Holdout - Scratch'), 'holdout_loc': dict(color='#f4a261', s=22, alpha=0.85, zorder=4, label='Holdout - Loc')}
    fig, ax = plt.subplots(figsize=(10, 7))
    for group_name in ['train_normal', 'test_normal', 'seen_defect', 'holdout_loc', 'holdout_scratch']:
        group_df = supervised_umap_df[supervised_umap_df['group'] == group_name]
        if group_df.empty:
            continue
        ax.scatter(group_df['umap_1'], group_df['umap_2'], **role_styles[group_name])
    ax.set_title('Supervised ViT-B/16 CLS embeddings by role')
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'supervised_umap_by_role.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved:', PLOTS_DIR / 'supervised_umap_by_role.png')


In [ ]:
if supervised_umap_df is None or supervised_umap_df.empty:
    print('[WARNING] No UMAP defect-type review was generated because the saved UMAP coordinates are unavailable.')
else:
    fig, ax = plt.subplots(figsize=(10, 7))
    normal_df = supervised_umap_df[supervised_umap_df['defect_type'] == 'normal']
    ax.scatter(normal_df['umap_1'], normal_df['umap_2'], s=4, alpha=0.2, color='#d1d5db', label='Normal')
    palette = plt.get_cmap('tab10')
    defect_types = [dtype for dtype in sorted(supervised_umap_df['defect_type'].dropna().unique()) if dtype != 'normal']
    for index, defect_type in enumerate(defect_types):
        defect_df = supervised_umap_df[supervised_umap_df['defect_type'] == defect_type]
        if defect_df.empty:
            continue
        marker = '*' if defect_type in {'Scratch', 'Loc'} else 'o'
        size = 48 if marker == '*' else 16
        ax.scatter(defect_df['umap_1'], defect_df['umap_2'], s=size, alpha=0.75, marker=marker, label=defect_type, color=palette(index % palette.N))
    ax.set_title('Supervised ViT-B/16 CLS embeddings by defect type')
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.legend(frameon=False, fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'supervised_umap_by_defect_type.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved:', PLOTS_DIR / 'supervised_umap_by_defect_type.png')


## UMAP Review

This notebook now keeps the saved CLS-embedding UMAP review together with the main supervised sweep analysis. If the UMAP exports are not present locally, the section logs a warning and moves on.


In [ ]:
UMAP_OUTPUT_DIR = ARTIFACT_DIR / 'umap'
LEGACY_UMAP_OUTPUT_DIR = EXPERIMENT_DIR / 'umap' / 'artifacts'
umap_coords_candidates = [UMAP_OUTPUT_DIR / 'umap_coords.csv', LEGACY_UMAP_OUTPUT_DIR / 'umap_coords.csv']
UMAP_COORDS_PATH = next((path for path in umap_coords_candidates if path.exists()), None)
supervised_umap_df = None
if UMAP_COORDS_PATH is None:
    print('[WARNING] No saved supervised-sweep UMAP coordinate CSV was found.')
    for candidate in umap_coords_candidates:
        print('  -', candidate)
else:
    supervised_umap_df = pd.read_csv(UMAP_COORDS_PATH)
    print(f'Loaded UMAP coordinates from {UMAP_COORDS_PATH}')
    display(supervised_umap_df.head())
    display(supervised_umap_df['group'].value_counts(dropna=False).rename('count').to_frame())


In [ ]:
if supervised_umap_df is None or supervised_umap_df.empty:
    print('[WARNING] No UMAP-by-role plot was generated because the saved UMAP coordinates are unavailable.')
else:
    role_styles = {'train_normal': dict(color='#d1d5db', s=4, alpha=0.25, zorder=1, label='Train normal'), 'test_normal': dict(color='#4d908e', s=8, alpha=0.4, zorder=2, label='Test normal'), 'seen_defect': dict(color='#277da1', s=14, alpha=0.65, zorder=3, label='Seen defect'), 'holdout_scratch': dict(color='#e63946', s=22, alpha=0.85, zorder=5, label='Holdout - Scratch'), 'holdout_loc': dict(color='#f4a261', s=22, alpha=0.85, zorder=4, label='Holdout - Loc')}
    fig, ax = plt.subplots(figsize=(10, 7))
    for group_name in ['train_normal', 'test_normal', 'seen_defect', 'holdout_loc', 'holdout_scratch']:
        group_df = supervised_umap_df[supervised_umap_df['group'] == group_name]
        if group_df.empty:
            continue
        ax.scatter(group_df['umap_1'], group_df['umap_2'], **role_styles[group_name])
    ax.set_title('Supervised ViT-B/16 CLS embeddings by role')
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'supervised_umap_by_role.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved:', PLOTS_DIR / 'supervised_umap_by_role.png')


In [ ]:
if supervised_umap_df is None or supervised_umap_df.empty:
    print('[WARNING] No UMAP defect-type review was generated because the saved UMAP coordinates are unavailable.')
else:
    fig, ax = plt.subplots(figsize=(10, 7))
    normal_df = supervised_umap_df[supervised_umap_df['defect_type'] == 'normal']
    ax.scatter(normal_df['umap_1'], normal_df['umap_2'], s=4, alpha=0.2, color='#d1d5db', label='Normal')
    palette = plt.get_cmap('tab10')
    defect_types = [dtype for dtype in sorted(supervised_umap_df['defect_type'].dropna().unique()) if dtype != 'normal']
    for index, defect_type in enumerate(defect_types):
        defect_df = supervised_umap_df[supervised_umap_df['defect_type'] == defect_type]
        if defect_df.empty:
            continue
        marker = '*' if defect_type in {'Scratch', 'Loc'} else 'o'
        size = 48 if marker == '*' else 16
        ax.scatter(defect_df['umap_1'], defect_df['umap_2'], s=size, alpha=0.75, marker=marker, label=defect_type, color=palette(index % palette.N))
    ax.set_title('Supervised ViT-B/16 CLS embeddings by defect type')
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.legend(frameon=False, fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'supervised_umap_by_defect_type.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved:', PLOTS_DIR / 'supervised_umap_by_defect_type.png')


## UMAP Review

This notebook now keeps the saved CLS-embedding UMAP review together with the main supervised sweep analysis. If the UMAP exports are not present locally, the section logs a warning and moves on.


In [ ]:
UMAP_OUTPUT_DIR = ARTIFACT_DIR / 'umap'
LEGACY_UMAP_OUTPUT_DIR = EXPERIMENT_DIR / 'umap' / 'artifacts'
umap_coords_candidates = [UMAP_OUTPUT_DIR / 'umap_coords.csv', LEGACY_UMAP_OUTPUT_DIR / 'umap_coords.csv']
UMAP_COORDS_PATH = next((path for path in umap_coords_candidates if path.exists()), None)
supervised_umap_df = None
if UMAP_COORDS_PATH is None:
    print('[WARNING] No saved supervised-sweep UMAP coordinate CSV was found.')
    for candidate in umap_coords_candidates:
        print('  -', candidate)
else:
    supervised_umap_df = pd.read_csv(UMAP_COORDS_PATH)
    print(f'Loaded UMAP coordinates from {UMAP_COORDS_PATH}')
    display(supervised_umap_df.head())
    display(supervised_umap_df['group'].value_counts(dropna=False).rename('count').to_frame())


In [ ]:
if supervised_umap_df is None or supervised_umap_df.empty:
    print('[WARNING] No UMAP-by-role plot was generated because the saved UMAP coordinates are unavailable.')
else:
    role_styles = {'train_normal': dict(color='#d1d5db', s=4, alpha=0.25, zorder=1, label='Train normal'), 'test_normal': dict(color='#4d908e', s=8, alpha=0.4, zorder=2, label='Test normal'), 'seen_defect': dict(color='#277da1', s=14, alpha=0.65, zorder=3, label='Seen defect'), 'holdout_scratch': dict(color='#e63946', s=22, alpha=0.85, zorder=5, label='Holdout - Scratch'), 'holdout_loc': dict(color='#f4a261', s=22, alpha=0.85, zorder=4, label='Holdout - Loc')}
    fig, ax = plt.subplots(figsize=(10, 7))
    for group_name in ['train_normal', 'test_normal', 'seen_defect', 'holdout_loc', 'holdout_scratch']:
        group_df = supervised_umap_df[supervised_umap_df['group'] == group_name]
        if group_df.empty:
            continue
        ax.scatter(group_df['umap_1'], group_df['umap_2'], **role_styles[group_name])
    ax.set_title('Supervised ViT-B/16 CLS embeddings by role')
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'supervised_umap_by_role.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved:', PLOTS_DIR / 'supervised_umap_by_role.png')


In [ ]:
if supervised_umap_df is None or supervised_umap_df.empty:
    print('[WARNING] No UMAP defect-type review was generated because the saved UMAP coordinates are unavailable.')
else:
    fig, ax = plt.subplots(figsize=(10, 7))
    normal_df = supervised_umap_df[supervised_umap_df['defect_type'] == 'normal']
    ax.scatter(normal_df['umap_1'], normal_df['umap_2'], s=4, alpha=0.2, color='#d1d5db', label='Normal')
    palette = plt.get_cmap('tab10')
    defect_types = [dtype for dtype in sorted(supervised_umap_df['defect_type'].dropna().unique()) if dtype != 'normal']
    for index, defect_type in enumerate(defect_types):
        defect_df = supervised_umap_df[supervised_umap_df['defect_type'] == defect_type]
        if defect_df.empty:
            continue
        marker = '*' if defect_type in {'Scratch', 'Loc'} else 'o'
        size = 48 if marker == '*' else 16
        ax.scatter(defect_df['umap_1'], defect_df['umap_2'], s=size, alpha=0.75, marker=marker, label=defect_type, color=palette(index % palette.N))
    ax.set_title('Supervised ViT-B/16 CLS embeddings by defect type')
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.legend(frameon=False, fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'supervised_umap_by_defect_type.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved:', PLOTS_DIR / 'supervised_umap_by_defect_type.png')
